In [65]:
import pandas as pd
import math

# Calcular o valor de cada contrapartida e o valor final a ser pago pelo empregador


def calcular_contrapartida(df, percentual_contrapartida, teto_percentual):
    """
    df: dataframe com colunas 'salario' e 'contribuicao_colaborador'
    percentual_contrapartida: ex. 0.5 para 50% da contribuição do colaborador
    teto_percentual: ex. 0.08 para 8% do salário (teto máximo da contrapartida)
    """
    df = df.copy()
    
    # Contrapartida bruta = % sobre o que o colaborador contribuiu
    contrapartida_bruta = df['contribuicao_colaborador'] * percentual_contrapartida
    
    # Teto = % do salário
    teto = df['salario'] * teto_percentual
    
    #  compara valor calculado e teto, pegando sempre o menor assim, a contrapartida nunca ultrapassa o teto.
    contrapartida_final = contrapartida_bruta.combine(teto, min)
    
    # Arredondamento de centavos para cima
    df['contrapartida_bruta'] = contrapartida_bruta.apply(lambda x: math.ceil(x * 100) / 100)
    df['teto_contrapartida'] = teto.apply(lambda x: math.ceil(x * 100) / 100)
    df['contrapartida_final'] = contrapartida_final.apply(lambda x: math.ceil(x * 100) / 100)

    return df


In [66]:
# Carrega as 3 abas da planilha de contribuições e faz o merge com as bases de colaboradores
contribuicoes = pd.read_excel('contribuicoes-agosto.xlsx', sheet_name=None)

# carrega as bases de colaboradores
df_vertex_base = pd.read_csv("colaboradores_vertex.csv")
df_sopro_base = pd.read_csv("colaboradores_sopro.csv")
df_aurora_base = pd.read_csv("colaboradores_aurora.csv")

# faz o merge das contribuições com as bases de colaboradores, usando o CPF como chave
df_vertex_completo = df_vertex_base.merge(
    contribuicoes['Vertex Logística'][['cpf', 'contribuicao_colaborador']],
    on='cpf', how='left'
)
df_sopro_completo = df_sopro_base.merge(
    contribuicoes['Sopro Alimentos'][['cpf', 'contribuicao_colaborador']],
    on='cpf', how='left'
)
df_aurora_completo = df_aurora_base.merge(
    contribuicoes['Aurora Clínicas'][['cpf', 'contribuicao_colaborador']],
    on='cpf', how='left'
)

# Aplica a função — cada empresa com seu percentual e teto
df_vertex_contrapartida = calcular_contrapartida(df_vertex_completo, percentual_contrapartida=0.5, teto_percentual=0.06)
df_sopro_contrapartida  = calcular_contrapartida(df_sopro_completo, percentual_contrapartida=1, teto_percentual=0.04)
df_aurora_contrapartida = calcular_contrapartida(df_aurora_completo, percentual_contrapartida=0.4, teto_percentual=0.05)

In [67]:
# exporta o resultado para um arquivo CSV
df_vertex_contrapartida.to_csv("contrapartida_vertex.csv", index=False)
df_sopro_contrapartida.to_csv("contrapartida_sopro.csv", index=False)
df_aurora_contrapartida.to_csv("contrapartida_aurora.csv", index=False)


In [68]:
# resumo geral das contribuições e contrapartidas por empresa
def resumo_empresa(df, nome_empresa):
    total_contribuicao = df['contribuicao_colaborador'].sum()
    total_contrapartida = df['contrapartida_final'].sum()
    total_geral = total_contribuicao + total_contrapartida
    
    return {
        'empresa': nome_empresa,
        'total_contribuicao': math.ceil(total_contribuicao * 100) / 100,
        'total_contrapartida': math.ceil(total_contrapartida * 100) / 100,
        'total_geral': math.ceil(total_geral * 100) / 100,
    }

resumos = [
    resumo_empresa(df_vertex_contrapartida, 'Vertex Logística'),
    resumo_empresa(df_sopro_contrapartida, 'Sopro Alimentos'),
    resumo_empresa(df_aurora_contrapartida, 'Aurora Clínicas'),
]

display(resumos)


[{'empresa': 'Vertex Logística',
  'total_contribuicao': 31744.33,
  'total_contrapartida': 15872.27,
  'total_geral': 47616.58},
 {'empresa': 'Sopro Alimentos',
  'total_contribuicao': 9576.1,
  'total_contrapartida': 9576.13,
  'total_geral': 19152.22},
 {'empresa': 'Aurora Clínicas',
  'total_contribuicao': 24720.74,
  'total_contrapartida': 9593.53,
  'total_geral': 34314.28}]

In [ ]:
# pega os valores dos boletos e compara com o total de cada empresa

# Boletos
boletos = (
    pd.read_csv('controle-boletos(Competência 08-2026).csv',
                sep=';', encoding='utf-8-sig', decimal=',', thousands='.')
      .dropna(how='all')
      .rename(columns={'nome': 'empresa'})
)

# Todas as empresas num único DF
df_final = pd.concat([df_vertex_contrapartida, df_sopro_contrapartida, df_aurora_contrapartida])

# Totais por empresa
resumo = (
    df_final.groupby('empresa')
            .agg(total_contribuicao=('contribuicao_colaborador', 'sum'),
                 total_contrapartida=('contrapartida_final', 'sum'))
            .assign(total_geral=lambda d: d['total_contribuicao'] + d['total_contrapartida'])
)

display(resumo)

# Comparação com os boletos
comparativo = resumo.merge(
    boletos[['empresa', 'valor', 'enviado ao cliente em']],
    on='empresa', how='left'
)

display(comparativo)

# Calcula a diferença entre o total geral e o valor do boleto
comparativo['diferenca'] = (comparativo['total_geral'] - comparativo['valor'])

print(comparativo.to_string(index=False))

# exporta o comparativo para um arquivo CSV
comparativo.to_csv("comparativo_boletos.csv", index=False)

,total_contribuicao,total_contrapartida,total_geral
empresa,,,
Aurora Clínicas,24720.74,9593.53,34314.27
Sopro Alimentos,9576.10,9576.12,19152.22
Vertex Logística,31744.32,15872.26,47616.58


,empresa,total_contribuicao,total_contrapartida,total_geral,valor,enviado ao cliente em
0,Aurora Clínicas,24720.74,9593.53,34314.27,34314.23,02/09/2026
1,Sopro Alimentos,9576.10,9576.12,19152.22,20190.40,01/09/2026
2,Vertex Logística,31744.32,15872.26,47616.58,48753.36,NaN


         empresa  total_contribuicao  total_contrapartida  total_geral    valor enviado ao cliente em  diferenca
 Aurora Clínicas            24720.74              9593.53     34314.27 34314.23            02/09/2026       0.04
 Sopro Alimentos             9576.10              9576.12     19152.22 20190.40            01/09/2026   -1038.18
Vertex Logística            31744.32             15872.26     47616.58 48753.36                   NaN   -1136.78


: 